# 01 · Cuenca y acuífero por municipio

Asigna a **cada municipio de México, por año (2016‑2024)**, las **cuencas hidrológicas** y
**acuíferos** de CONAGUA (tablas de *Disponibilidad*), con sus indicadores de disponibilidad y
sobreexplotación.

## Por qué NO es un join espacial

Las tablas que hay en el repo son de atributos, **sin geometría** (no traen lat/lon ni polígonos),
y además **no cubren todos los años** ni coinciden entre sí:

- `data/01_raw/cuencas/Disponibilidad en Cuencas hidrológicas_{2016,2020,2023}.xlsx`
- `data/01_raw/acuiferos/Disponibilidad de Acuiferos_{2015,2018,2020,2023}.xlsx`

Sin polígonos de cuencas/acuíferos no se puede calcular "está a X km". Lo que sí permiten las
**claves oficiales** es un cruce administrativo exacto:

| capa | clave | se une al municipio por |
|---|---|---|
| **acuífero** | `Clave Acuífero` (`EENN`) — los 2 primeros dígitos son la **entidad** | `idestado` (`CVE_ENT`) |
| **cuenca**   | `Clave Cuenca` (`RRNN`) — los 2 primeros son la **Región Hidrológica**; la tabla trae además la **RHA** (I–XIII) | **RHA** del municipio, tomada de `MunicipiosSequia.xlsx` (`CLV_OC`) |

Es decir: *"acuíferos de tu entidad"* y *"cuencas de tu Región Hidrológico‑Administrativa"* — contención
administrativa, no cercanía métrica. Si consigues los **shapefiles** de acuíferos/cuencas de CONAGUA,
la sección 7 hace el join espacial real (`cerca` = intersecta o ≤ `BUFFER_KM` km).

**Multi‑año**: como ninguna tabla cubre 2016‑2024 completo, cada año del panel toma la tabla
CONAGUA del **año disponible más próximo que sea ≤ ese año** (nunca se mira al futuro) — ver
sección 2.

## Salidas (`data/02_processed/`)

- `cuencas_acuifero_municipio.csv` — **1 fila por municipio × año** (2478 × 9 = 22 302 filas, ver sección 4):
  `idestado`, `idmunicipio`, `NOMGEO`, `anio`, `tiene_acuifero_sobreexplotado` (1/0),
  `tiene_cuenca_sin_disp` (1/0), `acuifero_disp`, `cuencas_disp`, `anio_fuente_acuifero`, `anio_fuente_cuenca`
- `cuencas_acuifero_mapa_anios.csv` — el mapeo año del panel → año CONAGUA usado
- `municipio_acuifero_largo.csv` / `municipio_cuenca_largo.csv` — 1 fila por par (municipio, acuífero|cuenca) por **año fuente**, con el detalle por rasgo


In [43]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option('display.max_columns', 80)

PROY      = Path('/Users/jaydymarchan/Desktop/causalidad')
RUTA_PROC = PROY / 'data/02_processed'
RUTA_MUNI_SHP    = PROY / 'data/01_raw/datos_lat_longitud/2024_1_00_MUN.shp'
RUTA_SEQUIA_XLSX = PROY / 'data/01_raw/datos_sequia/MunicipiosSequia.xlsx'      # crosswalk municipio -> RHA (CLV_OC)

# Carpetas con las tablas de disponibilidad CONAGUA. NO traen los mismos años entre sí ni
# cubren 2016-2024 completo -> cada año del panel se resuelve al año CONAGUA más próximo
# (ver sección 2: `anio_mas_proximo`).
RUTA_CUENCAS_DIR   = PROY / 'data/01_raw/cuencas'
RUTA_ACUIFEROS_DIR = PROY / 'data/01_raw/acuiferos'
ANIOS_MODELO = list(range(2016, 2025))   # años del panel de tabla_modelo.csv

# --- (opcional) polígonos para el join ESPACIAL de la sección 7 -------------
RUTA_CUENCAS_SHP   = None   # p.ej. PROY / 'data/01_raw/cuencas/cuencas.shp'
RUTA_ACUIFEROS_SHP = None   # p.ej. PROY / 'data/01_raw/acuiferos/acuiferos.shp'
BUFFER_KM   = 0.0
CRS_METRICO = 'EPSG:6372'   # Lambert Conformal Conic México (metros)


## 1 · Municipios + su Región Hidrológico‑Administrativa (RHA)

In [44]:
muni = gpd.read_file(RUTA_MUNI_SHP)[['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'NOMGEO', 'geometry']].to_crs(CRS_METRICO)
muni['idestado']    = muni['CVE_ENT'].astype(int)
muni['idmunicipio'] = muni['CVE_MUN'].astype(int)
muni = muni.sort_values('CVEGEO').reset_index(drop=True)

# crosswalk municipio -> RHA (CLV_OC) y Organismo de Cuenca, desde MunicipiosSequia.xlsx
xw = pd.read_excel(RUTA_SEQUIA_XLSX)[['CVE_ENT', 'CVE_MUN', 'ORG_CUENCA*', 'CLV_OC', 'CON_CUENCA']]
xw = xw.rename(columns={'ORG_CUENCA*': 'org_cuenca', 'CLV_OC': 'rha', 'CON_CUENCA': 'consejo_cuenca'})
xw['idestado']    = xw['CVE_ENT'].astype(int)
xw['idmunicipio'] = xw['CVE_MUN'].astype(int)
xw['rha'] = xw['rha'].astype(str).str.strip()

muni = muni.merge(xw[['idestado', 'idmunicipio', 'rha', 'org_cuenca', 'consejo_cuenca']],
                  on=['idestado', 'idmunicipio'], how='left')

print(f'{len(muni)} municipios | sin RHA: {muni["rha"].isna().sum()}')
print(muni['rha'].value_counts().sort_index().to_dict())
muni[['CVEGEO', 'idestado', 'idmunicipio', 'NOMGEO', 'rha', 'org_cuenca']].head()


2478 municipios | sin RHA: 0
{'I': 12, 'II': 78, 'III': 55, 'IV': 426, 'IX': 156, 'V': 363, 'VI': 131, 'VII': 85, 'VIII': 326, 'X': 461, 'XI': 145, 'XII': 126, 'XIII': 114}


,CVEGEO,idestado,idmunicipio,NOMGEO,rha,org_cuenca
0,01001,1,1,Aguascalientes,VIII,Lerma-Santiago-Pacífico
1,01002,1,2,Asientos,VIII,Lerma-Santiago-Pacífico
2,01003,1,3,Calvillo,VIII,Lerma-Santiago-Pacífico
3,01004,1,4,Cosío,VIII,Lerma-Santiago-Pacífico
4,01005,1,5,Jesús María,VIII,Lerma-Santiago-Pacífico


## 2 · Tablas de disponibilidad CONAGUA (multi-año, resueltas al año más próximo)

Las tablas de *Disponibilidad* de CONAGUA no se publican todos los años ni coinciden entre sí:

- **acuíferos** (`data/01_raw/acuiferos/Disponibilidad de Acuiferos_{año}.xlsx`): años disponibles **2015, 2018, 2020, 2023**
- **cuencas** (`data/01_raw/cuencas/Disponibilidad en Cuencas hidrológicas_{año}.xlsx`): años disponibles **2016, 2020, 2023**

Ninguna cubre 2016‑2024 completo ni coincide con la otra. Para cada año del panel se usa el **año
disponible más próximo que sea ≤ ese año** (nunca se mira al futuro; si no hay ninguno ≤, se usa
el más antiguo disponible). P. ej.: 2017 → acuífero 2015, cuenca 2016; 2022 → acuífero y cuenca 2020.

In [45]:
def _s(col):
    """Serie de texto SEGURA para encadenar .str.* — incluso si toda la columna es NaN.
    En pandas >= 3, .astype(str) sobre una columna 100% NaN NO la convierte a texto
    (se queda float64) y .str truena con 'Can only use .str accessor with string
    values, not floating'. Pasa exactamente con 'Condición' en los xlsx 2015/2018
    (CONAGUA no la publicó esos años)."""
    return col.fillna('').astype(str)


def _num(s):
    """'-95.76' / '1,234.5' / '' -> float."""
    return pd.to_numeric(_s(s).str.replace(',', '', regex=False).str.strip(), errors='coerce')


def _norm(s):
    return (_s(s).str.strip().str.lower()
            .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))


def _cargar_acuiferos(ruta):
    ac = pd.read_excel(ruta).rename(columns=lambda c: str(c).strip())
    ac = ac[ac['Clave Acuífero'].notna()].copy()
    condicion_disponible = ac['Condición'].notna().any()          # CONAGUA no publicó 'Condición' en 2015/2018
    ac['clave_acuifero'] = _s(ac['Clave Acuífero']).str.extract(r'(\d+)')[0].str.zfill(4)
    ac['idestado']       = ac['clave_acuifero'].str[:2].astype(int)          # los 2 primeros dígitos = entidad
    ac['rha']            = _s(ac['Clave RHA']).str.strip()
    ac['acuifero']       = _s(ac['Acuífero']).str.strip()
    ac['acuifero_disp']  = _num(ac['Disponibilidad'])                        # < 0  -> déficit
    ac['acuifero_sobreexplotado'] = _norm(ac['Condición']).eq('sobreexplotado')
    ac['acuifero_sin_disponibilidad'] = _norm(ac['Situación']).str.startswith('sin')
    out = ac[['clave_acuifero', 'acuifero', 'idestado', 'rha',
              'acuifero_disp', 'acuifero_sobreexplotado', 'acuifero_sin_disponibilidad']]
    out.attrs['condicion_disponible'] = bool(condicion_disponible)
    return out


def _cargar_cuencas(ruta):
    cu = pd.read_excel(ruta).rename(columns=lambda c: str(c).strip())
    cu = cu[cu['Clave Cuenca'].notna()].copy()
    _disp_col = next(c for c in cu.columns if c.lower().startswith('disponibilidad media anual'))
    cu['clave_cuenca'] = _s(cu['Clave Cuenca']).str.extract(r'(\d+)')[0].str.zfill(4)
    cu['rha']          = _s(cu['RHA']).str.strip()
    cu['region_hidro'] = _s(cu['RH']).str.strip()
    cu['cuenca']       = _s(cu['Nombre de la Cuenca']).str.strip()
    cu['cuenca_disp']  = _num(cu[_disp_col])
    cu['cuenca_sin_disponibilidad'] = _norm(cu['Clasificación']).str.startswith('sin')
    return cu[['clave_cuenca', 'cuenca', 'rha', 'region_hidro', 'cuenca_disp', 'cuenca_sin_disponibilidad']]


def _anios_en(carpeta):
    """Años detectados en los .xlsx de la carpeta (nombre termina en '{año}.xlsx')."""
    return sorted(int(m.group(1)) for p in carpeta.glob('*.xlsx')
                  if (m := re.search(r'(\d{4})\.xlsx$', p.name)))


ANIOS_ACUIFEROS = _anios_en(RUTA_ACUIFEROS_DIR)
ANIOS_CUENCAS   = _anios_en(RUTA_CUENCAS_DIR)
print(f'años disponibles -> acuíferos: {ANIOS_ACUIFEROS} | cuencas: {ANIOS_CUENCAS}')

ACU_POR_ANIO = {y: _cargar_acuiferos(RUTA_ACUIFEROS_DIR / f'Disponibilidad de Acuiferos_{y}.xlsx')
                for y in ANIOS_ACUIFEROS}
CUE_POR_ANIO = {y: _cargar_cuencas(RUTA_CUENCAS_DIR / f'Disponibilidad en Cuencas hidrológicas_{y}.xlsx')
                for y in ANIOS_CUENCAS}

for y, t in ACU_POR_ANIO.items():
    print(f'  acuíferos {y}: {len(t)}  (sobreexplotados {int(t.acuifero_sobreexplotado.sum())}, '
          f'sin disponibilidad {int(t.acuifero_sin_disponibilidad.sum())}, '
          f'"Condición" publicada: {t.attrs["condicion_disponible"]})')
for y, t in CUE_POR_ANIO.items():
    print(f'  cuencas   {y}: {len(t)}  (sin disponibilidad {int(t.cuenca_sin_disponibilidad.sum())})')

_sin_condicion = [y for y, t in ACU_POR_ANIO.items() if not t.attrs['condicion_disponible']]
if _sin_condicion:
    print(f'\n¡OJO! CONAGUA no publicó "Condición" (SOBREEXPLOTADO/SUBEXPLOTADO) en {_sin_condicion}:')
    print('  para los años del panel que caigan en esas fuentes, `tiene_acuifero_sobreexplotado`')
    print('  sale en 0 por falta de dato, NO porque no hubiera acuíferos sobreexplotados.')
    print('  Se marca en la columna `acuifero_condicion_ok` del panel final (sección 4).')


def anio_mas_proximo(anio_objetivo, anios_disponibles):
    """Año disponible <= anio_objetivo más cercano (nunca mira al futuro).
    Si ninguno es <=, usa el disponible más antiguo (no queda otra opción)."""
    anteriores = [a for a in anios_disponibles if a <= anio_objetivo]
    return max(anteriores) if anteriores else min(anios_disponibles)


mapa_anios = pd.DataFrame({'anio': ANIOS_MODELO})
mapa_anios['anio_fuente_acuifero'] = mapa_anios['anio'].apply(lambda a: anio_mas_proximo(a, ANIOS_ACUIFEROS))
mapa_anios['anio_fuente_cuenca']   = mapa_anios['anio'].apply(lambda a: anio_mas_proximo(a, ANIOS_CUENCAS))
mapa_anios['acuifero_condicion_ok'] = mapa_anios['anio_fuente_acuifero'].map(
    lambda y: ACU_POR_ANIO[y].attrs['condicion_disponible'])
print()
print('mapa año del panel -> año CONAGUA usado:')
print(mapa_anios.to_string(index=False))


años disponibles -> acuíferos: [2015, 2018, 2020, 2023] | cuencas: [2016, 2020, 2023]
  acuíferos 2015: 653  (sobreexplotados 0, sin disponibilidad 203, "Condición" publicada: False)
  acuíferos 2018: 653  (sobreexplotados 0, sin disponibilidad 245, "Condición" publicada: False)
  acuíferos 2020: 653  (sobreexplotados 111, sin disponibilidad 275, "Condición" publicada: True)
  acuíferos 2023: 653  (sobreexplotados 114, sin disponibilidad 286, "Condición" publicada: True)
  cuencas   2016: 757  (sin disponibilidad 108)
  cuencas   2020: 757  (sin disponibilidad 104)
  cuencas   2023: 757  (sin disponibilidad 104)

¡OJO! CONAGUA no publicó "Condición" (SOBREEXPLOTADO/SUBEXPLOTADO) en [2015, 2018]:
  para los años del panel que caigan en esas fuentes, `tiene_acuifero_sobreexplotado`
  sale en 0 por falta de dato, NO porque no hubiera acuíferos sobreexplotados.
  Se marca en la columna `acuifero_condicion_ok` del panel final (sección 4).

mapa año del panel -> año CONAGUA usado:
 anio  ani

## 3 · Asignación municipio ↔ acuífero (por entidad) y ↔ cuenca (por RHA), por año fuente

Se calcula **una vez por cada año fuente disponible** (no por año del panel, para no repetir
trabajo); la sección 4 arma el panel 2016‑2024 usando `mapa_anios` para saber qué año fuente le
toca a cada año del panel. Genera las tablas **largas** `municipio × acuífero` y `municipio × cuenca`
por año fuente.

In [46]:
muni_key = muni[['CVEGEO', 'idestado', 'idmunicipio', 'NOMGEO', 'rha']]

# municipio -> acuíferos de su entidad, para cada año fuente de acuíferos disponible
M_ACU_POR_ANIO = {
    y: (muni_key.merge(ACU, on='idestado', how='left')
                .sort_values(['CVEGEO', 'clave_acuifero']).reset_index(drop=True))
    for y, ACU in ACU_POR_ANIO.items()
}
# municipio -> cuencas de su RHA, para cada año fuente de cuencas disponible
M_CUE_POR_ANIO = {
    y: (muni_key.merge(CUE, on='rha', how='left')
                .sort_values(['CVEGEO', 'clave_cuenca']).reset_index(drop=True))
    for y, CUE in CUE_POR_ANIO.items()
}

for y, m in M_ACU_POR_ANIO.items():
    print(f'acuífero {y}: {len(m)} pares (media {len(m)/len(muni):.1f}/municipio)  |  '
          f'municipios sin acuífero: {m.groupby("CVEGEO")["clave_acuifero"].apply(lambda s: s.isna().all()).sum()}')
for y, m in M_CUE_POR_ANIO.items():
    print(f'cuenca   {y}: {len(m)} pares (media {len(m)/len(muni):.1f}/municipio)  |  '
          f'municipios sin cuenca  : {m.groupby("CVEGEO")["clave_cuenca"].apply(lambda s: s.isna().all()).sum()}')

M_ACU_POR_ANIO[ANIOS_ACUIFEROS[-1]].head()


acuífero 2015: 53062 pares (media 21.4/municipio)  |  municipios sin acuífero: 0
acuífero 2018: 53062 pares (media 21.4/municipio)  |  municipios sin acuífero: 0
acuífero 2020: 53062 pares (media 21.4/municipio)  |  municipios sin acuífero: 0
acuífero 2023: 53062 pares (media 21.4/municipio)  |  municipios sin acuífero: 0
cuenca   2016: 142049 pares (media 57.3/municipio)  |  municipios sin cuenca  : 0
cuenca   2020: 142049 pares (media 57.3/municipio)  |  municipios sin cuenca  : 0
cuenca   2023: 146456 pares (media 59.1/municipio)  |  municipios sin cuenca  : 0


,CVEGEO,idestado,idmunicipio,NOMGEO,rha_x,clave_acuifero,acuifero,rha_y,acuifero_disp,acuifero_sobreexplotado,acuifero_sin_disponibilidad
0,01001,1,1,Aguascalientes,VIII,0101,Valle de Aguascalientes,VIII,-95.76,True,True
1,01001,1,1,Aguascalientes,VIII,0102,Valle de Chicalote,VIII,-13.80,True,True
2,01001,1,1,Aguascalientes,VIII,0103,El Llano,VIII,-6.26,True,True
3,01001,1,1,Aguascalientes,VIII,0104,Venadero,VIII,-0.73,True,True
4,01001,1,1,Aguascalientes,VIII,0105,Valle de Calvillo,VIII,-17.71,True,True


## 4 · Panel municipio × año (2016‑2024)

`cuencas_acuifero_municipio.csv` — grano **municipio × año**: 2478 municipios × 9 años = **22 302 filas**, 11 columnas:

| columna | qué es |
|---|---|
| `idestado`, `idmunicipio`, `NOMGEO` | identificador del municipio |
| `anio` | año del panel (2016‑2024) |
| `tiene_acuifero_sobreexplotado` | **1/0** — ≥ 1 acuífero de su entidad con `Condición = SOBREEXPLOTADO`, **en el año fuente asignado a ese `anio`** |
| `tiene_cuenca_sin_disp` | **1/0** — ≥ 1 cuenca de su RHA con `Clasificación = Sin disponibilidad`, en el año fuente asignado |
| `acuifero_disp` | media de `Disponibilidad` de los acuíferos de su entidad, en el año fuente asignado |
| `cuencas_disp` | media de la disponibilidad media anual de agua superficial de las cuencas de su RHA, en el año fuente asignado |
| `acuifero_condicion_ok` | **1/0** — si es 0, CONAGUA **no publicó** la columna `Condición` en el año fuente de acuífero asignado (ocurre en 2015 y 2018) → `tiene_acuifero_sobreexplotado` sale en 0 **por falta de dato**, no porque no hubiera sobreexplotación. Afecta a los años del panel 2016‑2019. |
| `anio_fuente_acuifero` / `anio_fuente_cuenca` | qué año de CONAGUA se usó realmente para ese `anio` (ver `mapa_anios`, sección 2) |

> El valor solo cambia entre años del panel cuando cambia el **año fuente** asignado (p. ej. 2019→2020
> sube de acuífero 2018 a 2020); dentro de un mismo año fuente sigue siendo constante por
> entidad (acuífero) / RHA (cuenca). Las tablas largas `municipio_*_largo.csv` quedan por año fuente
> (no por año del panel) para no repetir filas.

In [47]:
# --- resumen por municipio, uno por cada año FUENTE disponible ---
def _resumen_acuifero(m_acu):
    g = m_acu.dropna(subset=['clave_acuifero']).groupby('CVEGEO')
    return pd.DataFrame({
        'tiene_acuifero_sobreexplotado': g['acuifero_sobreexplotado'].any(),
        'acuifero_disp': g['acuifero_disp'].mean().round(2),
    }).reset_index()


def _resumen_cuenca(m_cue):
    g = m_cue.dropna(subset=['clave_cuenca']).groupby('CVEGEO')
    return pd.DataFrame({
        'tiene_cuenca_sin_disp': g['cuenca_sin_disponibilidad'].any(),
        'cuencas_disp': g['cuenca_disp'].mean().round(2),
    }).reset_index()


RES_ACU_POR_ANIO = {y: _resumen_acuifero(m) for y, m in M_ACU_POR_ANIO.items()}
RES_CUE_POR_ANIO = {y: _resumen_cuenca(m) for y, m in M_CUE_POR_ANIO.items()}

# --- armar el panel municipio x año, usando mapa_anios para elegir el año fuente ---
paneles = []
for _, r in mapa_anios.iterrows():
    a, ac_y, cu_y = int(r['anio']), int(r['anio_fuente_acuifero']), int(r['anio_fuente_cuenca'])
    d = muni[['CVEGEO', 'idestado', 'idmunicipio', 'NOMGEO']].copy()
    d['anio'] = a
    d = d.merge(RES_ACU_POR_ANIO[ac_y], on='CVEGEO', how='left')
    d = d.merge(RES_CUE_POR_ANIO[cu_y], on='CVEGEO', how='left')
    d['anio_fuente_acuifero'] = ac_y
    d['anio_fuente_cuenca']   = cu_y
    d['acuifero_condicion_ok'] = bool(r['acuifero_condicion_ok'])   # False = 'Condición' no publicada esa fuente
    paneles.append(d)

resultado = pd.concat(paneles, ignore_index=True)
for c in ['tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp', 'acuifero_condicion_ok']:
    resultado[c] = resultado[c].fillna(False).astype(int)                        # -> 1 / 0

resultado = resultado[['idestado', 'idmunicipio', 'NOMGEO', 'anio',
                       'tiene_acuifero_sobreexplotado', 'tiene_cuenca_sin_disp',
                       'acuifero_disp', 'cuencas_disp', 'acuifero_condicion_ok',
                       'anio_fuente_acuifero', 'anio_fuente_cuenca']]
resultado.head(12)


,idestado,idmunicipio,NOMGEO,anio,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,anio_fuente_acuifero,anio_fuente_cuenca
0,1,1,Aguascalientes,2016,0,1,-29.64,249.77,0,2015,2016
1,1,2,Asientos,2016,0,1,-29.64,249.77,0,2015,2016
2,1,3,Calvillo,2016,0,1,-29.64,249.77,0,2015,2016
3,1,4,Cosío,2016,0,1,-29.64,249.77,0,2015,2016
4,1,5,Jesús María,2016,0,1,-29.64,249.77,0,2015,2016
5,1,6,Pabellón de Arteaga,2016,0,1,-29.64,249.77,0,2015,2016
6,1,7,Rincón de Romos,2016,0,1,-29.64,249.77,0,2015,2016
7,1,8,San José de Gracia,2016,0,1,-29.64,249.77,0,2015,2016
8,1,9,Tepezalá,2016,0,1,-29.64,249.77,0,2015,2016
9,1,10,El Llano,2016,0,1,-29.64,249.77,0,2015,2016


## 5 · Validación

In [48]:
n_esperado = len(muni) * len(ANIOS_MODELO)
print(f'filas: {len(resultado)}  (esperado {len(muni)} municipios x {len(ANIOS_MODELO)} años = {n_esperado})')
print('municipios:', resultado[['idestado', 'idmunicipio']].drop_duplicates().shape[0], '(esperado 2478)')
print('años      :', sorted(resultado['anio'].unique()))
print('llave (idestado,idmunicipio,anio) única:', resultado.duplicated(['idestado', 'idmunicipio', 'anio']).sum() == 0)
print('nulos     :', resultado.isna().sum().to_dict())
print()
print('mapa año del panel -> año CONAGUA usado:')
print(mapa_anios.to_string(index=False))
print()
_anios_sin_cond = sorted(resultado.loc[resultado['acuifero_condicion_ok'] == 0, 'anio'].unique())
print(f'años del panel con acuifero_condicion_ok=0 (tiene_acuifero_sobreexplotado NO confiable): {_anios_sin_cond}')
print()
print('tiene_acuifero_sobreexplotado por año (% de municipios):')
print(resultado.pivot_table(index='anio', values='tiene_acuifero_sobreexplotado', aggfunc='mean').round(3).to_string())
print('\ntiene_cuenca_sin_disp por año (% de municipios):')
print(resultado.pivot_table(index='anio', values='tiene_cuenca_sin_disp', aggfunc='mean').round(3).to_string())
print('\nmedia de disponibilidad por año:')
print(resultado.groupby('anio')[['acuifero_disp', 'cuencas_disp']].mean().round(2).to_string())
resultado.head(12)


filas: 22302  (esperado 2478 municipios x 9 años = 22302)
municipios: 2478 (esperado 2478)
años      : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
llave (idestado,idmunicipio,anio) única: True
nulos     : {'idestado': 0, 'idmunicipio': 0, 'NOMGEO': 0, 'anio': 0, 'tiene_acuifero_sobreexplotado': 0, 'tiene_cuenca_sin_disp': 0, 'acuifero_disp': 0, 'cuencas_disp': 0, 'acuifero_condicion_ok': 0, 'anio_fuente_acuifero': 0, 'anio_fuente_cuenca': 0}

mapa año del panel -> año CONAGUA usado:
 anio  anio_fuente_acuifero  anio_fuente_cuenca  acuifero_condicion_ok
 2016                  2015                2016                  False
 2017                  2015                2016                  False
 2018                  2018                2016                  False
 2019                  2018                2016                  False
 2020                  2020                2020         

,idestado,idmunicipio,NOMGEO,anio,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,anio_fuente_acuifero,anio_fuente_cuenca
0,1,1,Aguascalientes,2016,0,1,-29.64,249.77,0,2015,2016
1,1,2,Asientos,2016,0,1,-29.64,249.77,0,2015,2016
2,1,3,Calvillo,2016,0,1,-29.64,249.77,0,2015,2016
3,1,4,Cosío,2016,0,1,-29.64,249.77,0,2015,2016
4,1,5,Jesús María,2016,0,1,-29.64,249.77,0,2015,2016
5,1,6,Pabellón de Arteaga,2016,0,1,-29.64,249.77,0,2015,2016
6,1,7,Rincón de Romos,2016,0,1,-29.64,249.77,0,2015,2016
7,1,8,San José de Gracia,2016,0,1,-29.64,249.77,0,2015,2016
8,1,9,Tepezalá,2016,0,1,-29.64,249.77,0,2015,2016
9,1,10,El Llano,2016,0,1,-29.64,249.77,0,2015,2016


## 6 · Guardar

In [49]:
RUTA_PROC.mkdir(parents=True, exist_ok=True)

f = RUTA_PROC / 'cuencas_acuifero_municipio.csv'
resultado.to_csv(f, index=False, encoding='utf-8')
print('guardado ->', f, resultado.shape)

f_mapa = RUTA_PROC / 'cuencas_acuifero_mapa_anios.csv'
mapa_anios.to_csv(f_mapa, index=False, encoding='utf-8')
print('guardado ->', f_mapa, mapa_anios.shape)

# tablas largas: 1 fila por (municipio, rasgo, año FUENTE) -- no por año del panel, para no repetir
m_acu_largo = pd.concat([m.assign(anio_fuente_acuifero=y) for y, m in M_ACU_POR_ANIO.items()], ignore_index=True)
m_cue_largo = pd.concat([m.assign(anio_fuente_cuenca=y) for y, m in M_CUE_POR_ANIO.items()], ignore_index=True)
for nombre, largo in [('municipio_acuifero_largo', m_acu_largo), ('municipio_cuenca_largo', m_cue_largo)]:
    p = RUTA_PROC / f'{nombre}.csv'
    largo.to_csv(p, index=False, encoding='utf-8')
    print('guardado ->', p, largo.shape)

resultado.head(15)


guardado -> /Users/jaydymarchan/Desktop/causalidad/data/02_processed/cuencas_acuifero_municipio.csv (22302, 11)
guardado -> /Users/jaydymarchan/Desktop/causalidad/data/02_processed/cuencas_acuifero_mapa_anios.csv (9, 4)
guardado -> /Users/jaydymarchan/Desktop/causalidad/data/02_processed/municipio_acuifero_largo.csv (212248, 12)
guardado -> /Users/jaydymarchan/Desktop/causalidad/data/02_processed/municipio_cuenca_largo.csv (430554, 11)


,idestado,idmunicipio,NOMGEO,anio,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,anio_fuente_acuifero,anio_fuente_cuenca
0,1,1,Aguascalientes,2016,0,1,-29.64,249.77,0,2015,2016
1,1,2,Asientos,2016,0,1,-29.64,249.77,0,2015,2016
2,1,3,Calvillo,2016,0,1,-29.64,249.77,0,2015,2016
3,1,4,Cosío,2016,0,1,-29.64,249.77,0,2015,2016
4,1,5,Jesús María,2016,0,1,-29.64,249.77,0,2015,2016
5,1,6,Pabellón de Arteaga,2016,0,1,-29.64,249.77,0,2015,2016
6,1,7,Rincón de Romos,2016,0,1,-29.64,249.77,0,2015,2016
7,1,8,San José de Gracia,2016,0,1,-29.64,249.77,0,2015,2016
8,1,9,Tepezalá,2016,0,1,-29.64,249.77,0,2015,2016
9,1,10,El Llano,2016,0,1,-29.64,249.77,0,2015,2016


## 7 · (Opcional) Join espacial real con shapefiles de CONAGUA

Si consigues los polígonos de **acuíferos** (`Clave Acuífero` / `CLV_ACUIF`) y **cuencas
hidrológicas** (`Clave Cuenca`) de CONAGUA/SINA, pon las rutas en `RUTA_ACUIFEROS_SHP` /
`RUTA_CUENCAS_SHP` (celda de parámetros) y corre esta celda: reemplaza la asignación
administrativa por *"el municipio intersecta el polígono, o está a ≤ `BUFFER_KM` km"*,
y le pega los atributos de disponibilidad por la clave — tomados del **año fuente más reciente**
disponible (`max(ANIOS_ACUIFEROS)` / `max(ANIOS_CUENCAS)`); los polígonos no cambian de año a año,
así que esto no necesita repetirse por año del panel.

In [50]:
if RUTA_ACUIFEROS_SHP is None and RUTA_CUENCAS_SHP is None:
    print('Sin shapefiles: se mantiene la asignación administrativa de las secciones 3-6.')
else:
    def _cruzar_shp(capa_shp, col_clave, tabla_attrs, clave_attrs, prefijo):
        capa = gpd.read_file(capa_shp).to_crs(CRS_METRICO)
        capa['geometry'] = capa.geometry.buffer(0)
        capa = capa.rename(columns={col_clave: 'clave'})
        capa['clave'] = _s(capa['clave']).str.extract(r'(\d+)')[0].str.zfill(4)
        capa = capa.dissolve(by='clave', as_index=False)[['clave', 'geometry']]

        tope_m = BUFFER_KM * 1000.0
        pred = dict(predicate='dwithin', distance=tope_m) if tope_m > 0 else dict(predicate='intersects')
        pares = gpd.sjoin(muni[['CVEGEO', 'geometry']], capa, how='inner', **pred)[['CVEGEO', 'clave']]

        mm = pares.merge(muni[['CVEGEO', 'geometry']], on='CVEGEO').merge(
            capa.rename(columns={'geometry': '_g'}), on='clave')
        mm['dist_km'] = (gpd.GeoSeries(mm['geometry'].values, crs=CRS_METRICO)
                         .distance(gpd.GeoSeries(mm['_g'].values, crs=CRS_METRICO)) / 1000).round(3)
        largo = (mm[['CVEGEO', 'clave', 'dist_km']].drop_duplicates(['CVEGEO', 'clave'])
                 .merge(tabla_attrs, left_on='clave', right_on=clave_attrs, how='left'))
        return largo.sort_values(['CVEGEO', 'dist_km']).reset_index(drop=True)

    if RUTA_ACUIFEROS_SHP is not None:
        ACU_ULTIMO = ACU_POR_ANIO[max(ANIOS_ACUIFEROS)]
        m_acu_sp = _cruzar_shp(RUTA_ACUIFEROS_SHP, 'CLV_ACUIF', ACU_ULTIMO, 'clave_acuifero', 'acuifero')
        print(f'acuíferos (espacial, año {max(ANIOS_ACUIFEROS)}):', m_acu_sp.shape)
        display(m_acu_sp.head())
    if RUTA_CUENCAS_SHP is not None:
        CUE_ULTIMO = CUE_POR_ANIO[max(ANIOS_CUENCAS)]
        m_cue_sp = _cruzar_shp(RUTA_CUENCAS_SHP, 'CLV_CUENCA', CUE_ULTIMO, 'clave_cuenca', 'cuenca')
        print(f'cuencas (espacial, año {max(ANIOS_CUENCAS)}):', m_cue_sp.shape)
        display(m_cue_sp.head())
    print('\n-> re-aplica la sección 4 sobre m_acu_sp / m_cue_sp para regenerar el resumen.')


Sin shapefiles: se mantiene la asignación administrativa de las secciones 3-6.


## 8 · Uso en `02_join_data`

Ahora varía por **año** (usa el año fuente CONAGUA más próximo, ver `mapa_anios`) → se une por
municipio **y** año:

```python
ca = pd.read_csv('data/02_processed/cuencas_acuifero_municipio.csv')
df_modelo = df_modelo.merge(
    ca.drop(columns=['NOMGEO', 'anio_fuente_acuifero', 'anio_fuente_cuenca']),
    on=['idestado', 'idmunicipio', 'anio'], how='left',
)
```

Covariables de estrés hídrico (pre‑tratamiento): `tiene_acuifero_sobreexplotado`,
`tiene_cuenca_sin_disp`, `acuifero_disp`, `cuencas_disp`. Ojo con `acuifero_condicion_ok`: en
2016‑2019 vale 0 porque CONAGUA no publicó la columna `Condición` esos años fuente — no tratar
`tiene_acuifero_sobreexplotado = 0` como "sin sobreexplotación" en esas filas, sino como dato
faltante (considerar excluirlas del ajuste por esa variable, o usarla junto con `acuifero_condicion_ok`).


In [51]:
ca = pd.read_csv('data/02_processed/cuencas_acuifero_municipio.csv')
ca

,idestado,idmunicipio,NOMGEO,anio,tiene_acuifero_sobreexplotado,tiene_cuenca_sin_disp,acuifero_disp,cuencas_disp,acuifero_condicion_ok,anio_fuente_acuifero,anio_fuente_cuenca
0,1,1,Aguascalientes,2016,0,1,-29.64,249.77,0,2015,2016
1,1,2,Asientos,2016,0,1,-29.64,249.77,0,2015,2016
2,1,3,Calvillo,2016,0,1,-29.64,249.77,0,2015,2016
3,1,4,Cosío,2016,0,1,-29.64,249.77,0,2015,2016
4,1,5,Jesús María,2016,0,1,-29.64,249.77,0,2015,2016
...,...,...,...,...,...,...,...,...,...,...,...
22297,32,54,Villa Hidalgo,2024,1,0,-10.38,21.04,1,2023,2023
22298,32,55,Villanueva,2024,1,1,-10.38,95.20,1,2023,2023
22299,32,56,Zacatecas,2024,1,1,-10.38,95.20,1,2023,2023
22300,32,57,Trancoso,2024,1,0,-10.38,21.04,1,2023,2023
